Example of how to use codebase to analyse importance of weights in CNN model zoo

In [ ]:
# imports
import torch as th
import numpy as np
import matplotlib.pyplot as plt
from cnn_surgery.utils.load_dataset import load_dataset # this import is taking forever?

# Load Data
We load data from Unterthiner's small CNN model zoo. The metrics are enriched with per class accuracy for every model.

In [ ]:
dataset = 'fashion_mnist'
print('we are entering load_dataset()')
train, test, val = load_dataset(dataset, metrics_file='metrics_merged.csv', load_class_acc=True)
weights_train, outputs_train, configs_train = train
# Using validation set for test metrics in this example
weights_test, outputs_test, configs_test = val
train_class_accuracies = outputs_train[:, -10:]
test_class_accuracies = outputs_test[:, -10:]

# Get probe/lens model
We train an MLP regressor model to predict per class accuracy. If the regressor is of sufficiently good quality, i.e. it predicts class accuracy with a small enough MSE, we might learn what weights are important for predicitng per class accuracy from studying this probe/lens.

In [ ]:
from cnn_surgery.lenses.regressor_lens import get_regressor_lens
regressor_lens = get_regressor_lens(weights_train, train_class_accuracies, weights_test, test_class_accuracies)

Our regressor lens can make predictions on class accuracy

In [ ]:
# Define the CNN model index and retrieve its weights and class accuracy
CNN_index = 16  # Example index of a CNN model in the dataset
CNN_weights = weights_train[CNN_index]
CNN_class_accuracy = train_class_accuracies[CNN_index]

# Print model details
print(f"CNN model index: {CNN_index}")
print(f"Label class accuracy for CNN model {CNN_index}: {CNN_class_accuracy}")

# Predict class accuracy using the regressor lens
predicted_class_accuracy = regressor_lens.forward(
    th.tensor(CNN_weights, dtype=th.float32).unsqueeze(0)
).detach().numpy().flatten()
print(f"Predicted class accuracy: {predicted_class_accuracy}")

# Plot the predicted and actual class accuracies as a bar plot
plt.figure(figsize=(10, 6))
width = 0.35  # Width of the bars
x = np.arange(len(CNN_class_accuracy))  # Class indices

# Plot actual class accuracy
plt.bar(x - width / 2, CNN_class_accuracy, width, label="Actual Class Accuracy", alpha=0.7)

# Plot predicted class accuracy
plt.bar(x + width / 2, predicted_class_accuracy, width, label="Predicted Class Accuracy", alpha=0.7)

# Add plot details
plt.title(f"Class Accuracy for CNN Model {CNN_index}")
plt.xlabel("Class Index")
plt.ylabel("Accuracy")
plt.xticks(x)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Attribution analysis
By studying the regressor lens, we might learn which CNN weights are important for the regressor to make it's predicition. Let's try an attribution method; Integrated Gradients

In [ ]:
from captum.attr import IntegratedGradients

ig = IntegratedGradients(regressor_lens)

attributions = {}  # attributions for each class

for class_idx in range(10):
    attributions[class_idx] = ig.attribute(
        th.tensor(CNN_weights, dtype=th.float32).unsqueeze(0),
        target=class_idx,
).flatten().detach().numpy()

Let's plot the results (zoom in and scroll around)

In [ ]:
import plotly.graph_objects as go

# Create a grouped bar chart
fig = go.Figure()

for class_idx, class_attributions in attributions.items():
    mean_attributions = class_attributions
    fig.add_trace(go.Bar(
        x=list(range(len(mean_attributions))),
        y=mean_attributions,
        name=f'Class {class_idx}',
        width=0.9,
        opacity=0.7,
    ))

# Add layout details
fig.update_layout(
    title=f'Attributions Across input CNN weights for predicting accuracy for Each Class for {dataset} CNN Model {CNN_index}',
    xaxis_title='Feature Index (Weight Index)',
    yaxis_title='Attribution Value',
    barmode='overlay',  # Align bars for each dataset
    legend_title='Classes'
)

fig.show()